# ML1 Task 2 — Developer Salary Prediction
## Final Modeling Notebook for `outputs_v6`

This notebook uses the final corrected feature-engineering outputs from `02_feature_engineering_corrected_v6.ipynb` and compares only ML1-approved regression methods:

- Mean / median baseline
- Region-median baseline
- OLS
- Ridge
- LASSO
- Elastic Net
- KNN regression
- Linear SVR via `LinearSVR`
- SVR with RBF kernel
- SVR with polynomial kernel
- LASSO-selected SVR

The goal is to choose the final model based on **raw USD RMSE**, because the competition evaluates predictions for `annual.pay.usd` using RMSE.

### Important methodology

- Hyperparameters are tuned using 5-fold CV inside the internal training split.
- Model families are compared on the separate validation set.
- The internal test set is touched only once after choosing the final model.
- Predictions are trained on log salary, but evaluated after converting back to USD.
- Target variants are tested instead of assumed: raw log, winsorized log, floor-500 log, and drop-below-500 training rows.
- Calibration is tested because `exp(predicted_log_salary)` can underestimate USD salary.

Run this notebook after the feature-engineering notebook has produced the `outputs_v6/` folder.

In [1]:
# ===============================================================
# 1. Imports and global configuration
# ===============================================================
from __future__ import annotations

import json
import math
import os
import pickle
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, StratifiedKFold, ParameterGrid
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs_v6")
MODELING_DIR = Path("modeling_outputs_v6")
MODELING_DIR.mkdir(exist_ok=True)

# If runtime becomes too long, set this to True before running the search.
# FAST_MODE uses smaller grids but keeps the same logic.
FAST_MODE = True

# Number of folds for hyperparameter tuning inside the internal training split.
N_SPLITS = 5

# Raw salary prediction bounds for safety. We use loose bounds to avoid absurd values.
# These are not meant to artificially optimize; they only prevent numerical accidents.
PREDICTION_MIN_USD = 1.0
PREDICTION_MAX_USD = 1_000_000.0

print("Configuration loaded.")
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("MODELING_DIR:", MODELING_DIR.resolve())

Configuration loaded.
OUTPUT_DIR: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\outputs_v6
MODELING_DIR: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\modeling_outputs_v6


## 2. Load feature-engineering outputs

The feature-engineering notebook created two matrices:

- `FULL`: rich 303-feature matrix for regularized and nonlinear models.
- `OLS_SAFE`: reduced 108-feature matrix for OLS baseline.

We also load several target variants. The validation and internal-test targets remain raw salaries for honest RMSE evaluation.

In [2]:
# ===============================================================
# 2. Load data generated by feature engineering v6
# ===============================================================

def read_series(path: Path) -> pd.Series:
    """Read a one-column CSV as a Series."""
    obj = pd.read_csv(path)
    if obj.shape[1] != 1:
        raise ValueError(f"Expected one column in {path}, got {obj.shape[1]} columns.")
    return obj.iloc[:, 0]

required_files = [
    "X_train_full_scaled.csv",
    "X_val_full_scaled.csv",
    "X_internal_test_full_scaled.csv",
    "X_kaggle_full_scaled.csv",
    "X_train_ols_safe_scaled.csv",
    "X_val_ols_safe_scaled.csv",
    "X_internal_test_ols_safe_scaled.csv",
    "y_train_log_raw.csv",
    "y_train_log_winsorized.csv",
    "y_train_log_floor500.csv",
    "y_train_raw.csv",
    "y_val_raw.csv",
    "y_internal_test_raw.csv",
    "y_val_log_raw.csv",
    "y_internal_test_log_raw.csv",
    "train_keep_salary_ge_500_mask.csv",
    "train_original_indices.csv",
    "val_original_indices.csv",
    "internal_test_original_indices.csv",
    "kaggle_ids.csv",
    "feature_engineering_summary.json",
]

missing = [f for f in required_files if not (OUTPUT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        "Missing files in outputs_v6. Run the final feature engineering notebook first.\n"
        f"Missing: {missing}"
    )

# Full feature matrices.
X_train_full = pd.read_csv(OUTPUT_DIR / "X_train_full_scaled.csv")
X_val_full = pd.read_csv(OUTPUT_DIR / "X_val_full_scaled.csv")
X_internal_test_full = pd.read_csv(OUTPUT_DIR / "X_internal_test_full_scaled.csv")
X_kaggle_full = pd.read_csv(OUTPUT_DIR / "X_kaggle_full_scaled.csv")

# OLS-safe matrices.
X_train_ols = pd.read_csv(OUTPUT_DIR / "X_train_ols_safe_scaled.csv")
X_val_ols = pd.read_csv(OUTPUT_DIR / "X_val_ols_safe_scaled.csv")
X_internal_test_ols = pd.read_csv(OUTPUT_DIR / "X_internal_test_ols_safe_scaled.csv")

# Target variants for internal training rows.
y_train_log_raw = read_series(OUTPUT_DIR / "y_train_log_raw.csv")
y_train_log_winsorized = read_series(OUTPUT_DIR / "y_train_log_winsorized.csv")
y_train_log_floor500 = read_series(OUTPUT_DIR / "y_train_log_floor500.csv")
y_train_raw = read_series(OUTPUT_DIR / "y_train_raw.csv")

# Validation and internal test targets.
y_val_raw = read_series(OUTPUT_DIR / "y_val_raw.csv")
y_val_log_raw = read_series(OUTPUT_DIR / "y_val_log_raw.csv")
y_internal_test_raw = read_series(OUTPUT_DIR / "y_internal_test_raw.csv")
y_internal_test_log_raw = read_series(OUTPUT_DIR / "y_internal_test_log_raw.csv")

# Training mask for the drop-below-500 experiment.
train_keep_salary_ge_500 = read_series(OUTPUT_DIR / "train_keep_salary_ge_500_mask.csv").astype(bool)

# IDs for Kaggle submission.
kaggle_ids = read_series(OUTPUT_DIR / "kaggle_ids.csv")

# Feature engineering summary.
with open(OUTPUT_DIR / "feature_engineering_summary.json", "r", encoding="utf-8") as f:
    feature_summary = json.load(f)

print("Loaded feature-engineering outputs.")
print("X_train_full:", X_train_full.shape)
print("X_val_full:", X_val_full.shape)
print("X_internal_test_full:", X_internal_test_full.shape)
print("X_kaggle_full:", X_kaggle_full.shape)
print("X_train_ols:", X_train_ols.shape)
print("X_val_ols:", X_val_ols.shape)
print("X_internal_test_ols:", X_internal_test_ols.shape)
print("Low-salary training rows dropped in drop_lt500 variant:", int((~train_keep_salary_ge_500).sum()))

Loaded feature-engineering outputs.
X_train_full: (1758, 303)
X_val_full: (377, 303)
X_internal_test_full: (377, 303)
X_kaggle_full: (628, 303)
X_train_ols: (1758, 108)
X_val_ols: (377, 108)
X_internal_test_ols: (377, 108)
Low-salary training rows dropped in drop_lt500 variant: 57


In [3]:
# ===============================================================
# 3. Integrity checks
# ===============================================================

def check_matrix(name: str, X: pd.DataFrame) -> Dict[str, int]:
    return {
        "matrix": name,
        "rows": X.shape[0],
        "cols": X.shape[1],
        "nans": int(X.isna().sum().sum()),
        "duplicate_columns": int(X.columns.duplicated().sum()),
        "zero_variance_cols": int((X.nunique(dropna=False) <= 1).sum()),
    }

checks = pd.DataFrame([
    check_matrix("X_train_full", X_train_full),
    check_matrix("X_val_full", X_val_full),
    check_matrix("X_internal_test_full", X_internal_test_full),
    check_matrix("X_kaggle_full", X_kaggle_full),
    check_matrix("X_train_ols", X_train_ols),
    check_matrix("X_val_ols", X_val_ols),
    check_matrix("X_internal_test_ols", X_internal_test_ols),
])

display(checks)

assert checks["nans"].sum() == 0, "There are NaNs in feature matrices."
assert checks["duplicate_columns"].sum() == 0, "There are duplicate columns."
assert checks["zero_variance_cols"].sum() == 0, "There are zero-variance columns."
assert len(X_train_full) == len(y_train_log_raw) == len(y_train_raw), "Train X/y length mismatch."
assert len(X_val_full) == len(y_val_raw), "Validation X/y length mismatch."
assert len(X_internal_test_full) == len(y_internal_test_raw), "Internal test X/y length mismatch."
print("All integrity checks passed.")

,matrix,rows,cols,nans,duplicate_columns,zero_variance_cols
0,X_train_full,1758,303,0,0,0
1,X_val_full,377,303,0,0,0
2,X_internal_test_full,377,303,0,0,0
3,X_kaggle_full,628,303,0,0,0
4,X_train_ols,1758,108,0,0,0
5,X_val_ols,377,108,0,0,0
6,X_internal_test_ols,377,108,0,0,0


All integrity checks passed.


## 3. Recover region information for the region baseline and stratified CV

The feature matrices are numeric, so we reload the original row indices to recover region labels from `train.csv`. This is used only for:

- a simple region-median baseline,
- region-stratified cross-validation folds.

In [4]:
# ===============================================================
# 4. Recover original split regions
# ===============================================================

def normalize_text_value(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    return " ".join(value.split())

train_raw_full = pd.read_csv("train.csv")
train_raw_full["region"] = train_raw_full["region"].map(normalize_text_value)

train_indices = read_series(OUTPUT_DIR / "train_original_indices.csv").astype(int)
val_indices = read_series(OUTPUT_DIR / "val_original_indices.csv").astype(int)
internal_test_indices = read_series(OUTPUT_DIR / "internal_test_original_indices.csv").astype(int)

region_train = train_raw_full.loc[train_indices, "region"].reset_index(drop=True)
region_val = train_raw_full.loc[val_indices, "region"].reset_index(drop=True)
region_internal_test = train_raw_full.loc[internal_test_indices, "region"].reset_index(drop=True)

print("Region distribution in internal training split:")
display(region_train.value_counts().to_frame("count").head(20))

Region distribution in internal training split:


,count
region,
R17,776
R11,303
R13,148
R18,86
R03,80
R05,56
R06,53
R04,42
R16,42


## 4. Metrics, calibration and target variants

We train models on log salary. For evaluation, predictions are transformed back to USD and compared with raw salary using RMSE.

We test three calibration options:

1. `none`: `exp(pred_log)`
2. `train_smearing`: Duan-style smearing factor based on training residuals
3. `validation_multiplier`: simple multiplier selected on the validation set

The validation multiplier is allowed as part of model selection, but the internal test is then used only once to evaluate the selected complete modeling strategy.

In [5]:
# ===============================================================
# 5. Metrics, calibration, and target variants
# ===============================================================

def rmse(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_exp_to_usd(pred_log: np.ndarray, factor: float = 1.0) -> np.ndarray:
    """Convert log predictions to positive USD predictions with broad safety clipping."""
    pred = np.exp(np.asarray(pred_log, dtype=float)) * factor
    pred = np.clip(pred, PREDICTION_MIN_USD, PREDICTION_MAX_USD)
    return pred


def usd_rmse_from_log(y_true_raw: pd.Series, pred_log: np.ndarray, factor: float = 1.0) -> float:
    return rmse(y_true_raw, safe_exp_to_usd(pred_log, factor=factor))


def optimize_validation_multiplier(
    y_val_raw: pd.Series,
    pred_val_log: np.ndarray,
    grid: Optional[np.ndarray] = None,
) -> Tuple[float, float]:
    """Return best multiplicative factor and validation RMSE."""
    if grid is None:
        # Fairly broad but not absurd. Refine later if needed.
        grid = np.linspace(0.80, 1.35, 112)
    scores = []
    for factor in grid:
        scores.append(usd_rmse_from_log(y_val_raw, pred_val_log, factor=factor))
    best_idx = int(np.argmin(scores))
    return float(grid[best_idx]), float(scores[best_idx])


def calibration_report(
    model,
    X_train: pd.DataFrame,
    y_train_log: pd.Series,
    y_train_raw: pd.Series,
    X_val: pd.DataFrame,
    y_val_raw: pd.Series,
) -> Dict[str, Any]:
    """Fit model, predict validation, and evaluate calibration options."""
    fitted = clone(model)
    fitted.fit(X_train, y_train_log)

    pred_train_log = np.asarray(fitted.predict(X_train), dtype=float)
    pred_val_log = np.asarray(fitted.predict(X_val), dtype=float)

    residuals_log = np.asarray(y_train_log, dtype=float) - pred_train_log
    smearing_factor = float(np.mean(np.exp(residuals_log)))

    rmse_none = usd_rmse_from_log(y_val_raw, pred_val_log, factor=1.0)
    rmse_smearing = usd_rmse_from_log(y_val_raw, pred_val_log, factor=smearing_factor)
    val_factor, rmse_val_multiplier = optimize_validation_multiplier(y_val_raw, pred_val_log)

    candidates = {
        "none": (1.0, rmse_none),
        "train_smearing": (smearing_factor, rmse_smearing),
        "validation_multiplier": (val_factor, rmse_val_multiplier),
    }
    best_correction = min(candidates, key=lambda key: candidates[key][1])
    best_factor, best_rmse = candidates[best_correction]

    return {
        "fitted_model": fitted,
        "pred_val_log": pred_val_log,
        "pred_train_log": pred_train_log,
        "smearing_factor": smearing_factor,
        "validation_multiplier": val_factor,
        "val_rmse_none": rmse_none,
        "val_rmse_smearing": rmse_smearing,
        "val_rmse_validation_multiplier": rmse_val_multiplier,
        "best_correction": best_correction,
        "best_factor": best_factor,
        "best_val_rmse": best_rmse,
    }


def get_target_variant_data(
    X_train: pd.DataFrame,
    target_variant: str,
    region_labels: pd.Series,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    """
    Return X, y_log, y_raw, and region labels for a target variant.

    Variants:
    - raw: raw log target
    - winsorized: train-only winsorized log target
    - floor500: salaries below 500 are floored to 500 for training target
    - drop_lt500: remove training rows with salary below 500
    """
    if target_variant == "raw":
        mask = pd.Series(True, index=X_train.index)
        y_log = y_train_log_raw.copy()
        y_raw = y_train_raw.copy()
    elif target_variant == "winsorized":
        mask = pd.Series(True, index=X_train.index)
        y_log = y_train_log_winsorized.copy()
        y_raw = y_train_raw.copy()
    elif target_variant == "floor500":
        mask = pd.Series(True, index=X_train.index)
        y_log = y_train_log_floor500.copy()
        y_raw = y_train_raw.clip(lower=500).copy()
    elif target_variant == "drop_lt500":
        mask = pd.Series(train_keep_salary_ge_500.to_numpy(), index=X_train.index).astype(bool)
        y_log = y_train_log_raw.loc[mask].reset_index(drop=True)
        y_raw = y_train_raw.loc[mask].reset_index(drop=True)
    else:
        raise ValueError(f"Unknown target variant: {target_variant}")

    X_out = X_train.loc[mask].reset_index(drop=True)
    region_out = region_labels.loc[mask].reset_index(drop=True)
    y_log = y_log.reset_index(drop=True)
    y_raw = y_raw.reset_index(drop=True)
    return X_out, y_log, y_raw, region_out


def make_region_strata(region_series: pd.Series, min_count: int = 10) -> pd.Series:
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def make_cv_splitter(region_labels: pd.Series, n_splits: int = N_SPLITS):
    """Use StratifiedKFold by region when feasible; otherwise fall back to KFold."""
    strata = make_region_strata(region_labels)
    counts = strata.value_counts()
    if len(counts) >= 2 and counts.min() >= n_splits:
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE), strata
    return KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE), None

print("Metric and target-variant helpers ready.")

Metric and target-variant helpers ready.


## 5. Baselines

Baselines are important because they tell us whether advanced models are actually adding value.

We test:

- mean salary baseline,
- median salary baseline,
- region-median baseline.

In [6]:
# ===============================================================
# 6. Baselines
# ===============================================================

baseline_records = []

def add_baseline(name: str, pred_val: np.ndarray, pred_test: Optional[np.ndarray] = None):
    record = {
        "model_name": name,
        "model_family": "baseline",
        "feature_set": "none",
        "target_variant": "raw_salary",
        "params": "{}",
        "cv_rmse_mean": np.nan,
        "cv_rmse_std": np.nan,
        "val_rmse_none": rmse(y_val_raw, pred_val),
        "val_rmse_smearing": np.nan,
        "val_rmse_validation_multiplier": np.nan,
        "best_correction": "none",
        "best_factor": 1.0,
        "best_val_rmse": rmse(y_val_raw, pred_val),
        "n_features": 0,
        "n_train_rows": len(y_train_raw),
        "fit_seconds": 0.0,
    }
    baseline_records.append(record)

# Global mean and median baselines.
add_baseline("Mean salary baseline", np.repeat(y_train_raw.mean(), len(y_val_raw)))
add_baseline("Median salary baseline", np.repeat(y_train_raw.median(), len(y_val_raw)))

# Region-median baseline using training split only.
region_salary_train = pd.DataFrame({"region": region_train, "salary": y_train_raw})
region_medians = region_salary_train.groupby("region")["salary"].median()
global_median = float(y_train_raw.median())
pred_region_val = region_val.map(region_medians).fillna(global_median).to_numpy()
add_baseline("Region-median baseline", pred_region_val)

baseline_df = pd.DataFrame(baseline_records).sort_values("best_val_rmse")
display(baseline_df[["model_name", "best_val_rmse"]])

,model_name,best_val_rmse
2,Region-median baseline,43819.499579
1,Median salary baseline,44675.864782
0,Mean salary baseline,44735.505271


## 6. Cross-validation and grid-search functions

This function evaluates each hyperparameter combination using:

- 5-fold CV inside the internal training split,
- validation-set RMSE after back-transforming to USD,
- calibration options.

A note about CV: because we use precomputed `outputs_v6` matrices, preprocessing was already fitted on the full internal training split. Therefore CV is used mainly for hyperparameter guidance. The separate validation set remains the primary model-selection set.

In [7]:
# ===============================================================
# 7. Cross-validation and grid search
# ===============================================================

def cv_usd_rmse(
    model,
    X: pd.DataFrame,
    y_log: pd.Series,
    y_raw: pd.Series,
    region_labels: pd.Series,
    n_splits: int = N_SPLITS,
) -> Tuple[float, float]:
    """Manual CV using USD RMSE after exp back-transform."""
    splitter, strata = make_cv_splitter(region_labels, n_splits=n_splits)
    scores = []

    if strata is None:
        split_iter = splitter.split(X)
    else:
        split_iter = splitter.split(X, strata)

    for tr_idx, va_idx in split_iter:
        fold_model = clone(model)
        fold_model.fit(X.iloc[tr_idx], y_log.iloc[tr_idx])
        pred_log = np.asarray(fold_model.predict(X.iloc[va_idx]), dtype=float)
        score = usd_rmse_from_log(y_raw.iloc[va_idx], pred_log, factor=1.0)
        scores.append(score)

    return float(np.mean(scores)), float(np.std(scores, ddof=1))


def evaluate_candidate(
    model,
    model_name: str,
    model_family: str,
    feature_set: str,
    target_variant: str,
    params: Dict[str, Any],
    X_train_base: pd.DataFrame,
    X_val_base: pd.DataFrame,
    region_labels: pd.Series,
) -> Dict[str, Any]:
    """Evaluate one model candidate."""
    Xtr, ylog, yraw, reg = get_target_variant_data(X_train_base, target_variant, region_labels)
    n_features = Xtr.shape[1]
    n_train_rows = Xtr.shape[0]

    start = time.time()
    cv_mean, cv_std = cv_usd_rmse(model, Xtr, ylog, yraw, reg)
    report = calibration_report(model, Xtr, ylog, yraw, X_val_base, y_val_raw)
    elapsed = time.time() - start

    return {
        "model_name": model_name,
        "model_family": model_family,
        "feature_set": feature_set,
        "target_variant": target_variant,
        "params": json.dumps(params),
        "cv_rmse_mean": cv_mean,
        "cv_rmse_std": cv_std,
        "val_rmse_none": report["val_rmse_none"],
        "val_rmse_smearing": report["val_rmse_smearing"],
        "val_rmse_validation_multiplier": report["val_rmse_validation_multiplier"],
        "best_correction": report["best_correction"],
        "best_factor": report["best_factor"],
        "best_val_rmse": report["best_val_rmse"],
        "n_features": n_features,
        "n_train_rows": n_train_rows,
        "fit_seconds": elapsed,
    }


def run_grid(
    model_family: str,
    estimator_factory,
    param_grid: List[Dict[str, Any]],
    X_train_base: pd.DataFrame,
    X_val_base: pd.DataFrame,
    feature_set: str,
    target_variants: List[str],
    region_labels: pd.Series,
    results: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Run a grid and append records to results."""
    combinations = list(ParameterGrid(param_grid))
    total = len(combinations) * len(target_variants)
    counter = 0
    print(f"\n=== {model_family}: {total} candidates ===")

    for target_variant in target_variants:
        for params in combinations:
            counter += 1
            model = estimator_factory(**params)
            model_name = f"{model_family} | {target_variant} | {params}"
            print(f"[{counter:03d}/{total:03d}] {model_name}")
            try:
                record = evaluate_candidate(
                    model=model,
                    model_name=model_name,
                    model_family=model_family,
                    feature_set=feature_set,
                    target_variant=target_variant,
                    params=params,
                    X_train_base=X_train_base,
                    X_val_base=X_val_base,
                    region_labels=region_labels,
                )
                results.append(record)
                print(f"    CV RMSE: {record['cv_rmse_mean']:,.0f} | Val RMSE: {record['best_val_rmse']:,.0f} | correction={record['best_correction']}")
            except Exception as exc:
                print(f"    FAILED: {exc}")
                results.append({
                    "model_name": model_name,
                    "model_family": model_family,
                    "feature_set": feature_set,
                    "target_variant": target_variant,
                    "params": json.dumps(params),
                    "cv_rmse_mean": np.nan,
                    "cv_rmse_std": np.nan,
                    "val_rmse_none": np.nan,
                    "val_rmse_smearing": np.nan,
                    "val_rmse_validation_multiplier": np.nan,
                    "best_correction": "failed",
                    "best_factor": np.nan,
                    "best_val_rmse": np.nan,
                    "n_features": X_train_base.shape[1],
                    "n_train_rows": len(X_train_base),
                    "fit_seconds": np.nan,
                    "error": str(exc),
                })

            # Save after each candidate so progress is not lost.
            pd.DataFrame(results).to_csv(MODELING_DIR / "model_search_results_partial.csv", index=False)

    return results

print("Grid-search helpers ready.")

Grid-search helpers ready.


## 7. Define model grids

The grids are intentionally focused. They are broad enough to compare ML1 models, but not so huge that the notebook becomes impossible to run.

Important: `SVR(kernel="linear")` is intentionally **not** used because it can be very slow. We use `LinearSVR` instead.

In [8]:
# ===============================================================
# 8. Model grids
# ===============================================================

TARGET_VARIANTS_ALL = ["raw", "winsorized", "floor500", "drop_lt500"]
TARGET_VARIANTS_MAIN = TARGET_VARIANTS_ALL
TARGET_VARIANTS_FAST = ["raw", "winsorized"]

if FAST_MODE:
    print("FAST_MODE is ON: using smaller grids.")
    target_variants_linear = TARGET_VARIANTS_FAST
    target_variants_svr = TARGET_VARIANTS_FAST
else:
    target_variants_linear = TARGET_VARIANTS_MAIN
    target_variants_svr = TARGET_VARIANTS_MAIN

# Model grids.
ridge_grid = [{"alpha": [0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0]}]

lasso_grid = [{"alpha": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03]}]

elastic_grid = [{
    "alpha": [0.0003, 0.001, 0.003, 0.01],
    "l1_ratio": [0.2, 0.5, 0.8],
}]

knn_grid = [{
    "n_neighbors": [5, 10, 20, 40, 80] if not FAST_MODE else [10, 40],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}]

linear_svr_grid = [{
    "C": [0.05, 0.1, 0.3, 1.0, 3.0] if not FAST_MODE else [0.1, 1.0],
    "epsilon": [0.05, 0.1, 0.2],
}]

svr_rbf_grid = [{
    # Focused grid to keep runtime realistic while still testing regularization and locality.
    "C": [0.3, 1.0, 3.0, 10.0] if not FAST_MODE else [1.0, 3.0],
    "gamma": ["scale", 0.001, 0.0003] if not FAST_MODE else ["scale", 0.0003],
    "epsilon": [0.1, 0.2],
}]

svr_poly_grid = [{
    # Polynomial SVR can be slow, so this grid is intentionally smaller than RBF.
    "C": [0.3, 1.0, 3.0] if not FAST_MODE else [1.0],
    "degree": [2, 3],
    "gamma": ["scale", 0.0003] if not FAST_MODE else ["scale"],
    "epsilon": [0.1],
    "coef0": [1.0],
}]

print("Grid sizes:")
print("Ridge:", len(list(ParameterGrid(ridge_grid))) * len(target_variants_linear))
print("LASSO:", len(list(ParameterGrid(lasso_grid))) * len(target_variants_linear))
print("Elastic Net:", len(list(ParameterGrid(elastic_grid))) * len(target_variants_linear))
print("KNN:", len(list(ParameterGrid(knn_grid))) * len(["raw", "winsorized"]))
print("LinearSVR:", len(list(ParameterGrid(linear_svr_grid))) * len(target_variants_linear))
print("SVR RBF:", len(list(ParameterGrid(svr_rbf_grid))) * len(target_variants_svr))
print("SVR Poly:", len(list(ParameterGrid(svr_poly_grid))) * len(target_variants_svr))

FAST_MODE is ON: using smaller grids.
Grid sizes:
Ridge: 16
LASSO: 12
Elastic Net: 24
KNN: 16
LinearSVR: 12
SVR RBF: 16
SVR Poly: 4


## 8. Run model search

This cell can take time, especially for RBF and polynomial SVR. If needed, set `FAST_MODE = True` in the first cell and rerun the notebook.

The partial results are saved after every candidate to `modeling_outputs_v6/model_search_results_partial.csv`.

In [9]:
# ===============================================================
# 9. Run baseline and main model search
# ===============================================================

all_results = baseline_records.copy()

# OLS baseline on OLS_SAFE. We test raw and winsorized target only; OLS is a baseline, not the expected winner.
all_results = run_grid(
    model_family="OLS",
    estimator_factory=lambda **params: LinearRegression(**params),
    param_grid=[{}],
    X_train_base=X_train_ols,
    X_val_base=X_val_ols,
    feature_set="OLS_SAFE",
    target_variants=["raw", "winsorized", "floor500", "drop_lt500"],
    region_labels=region_train,
    results=all_results,
)

all_results = run_grid(
    model_family="Ridge",
    estimator_factory=lambda **params: Ridge(**params, random_state=RANDOM_STATE),
    param_grid=ridge_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_linear,
    region_labels=region_train,
    results=all_results,
)

all_results = run_grid(
    model_family="LASSO",
    estimator_factory=lambda **params: Lasso(**params, max_iter=50000, random_state=RANDOM_STATE),
    param_grid=lasso_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_linear,
    region_labels=region_train,
    results=all_results,
)

all_results = run_grid(
    model_family="ElasticNet",
    estimator_factory=lambda **params: ElasticNet(**params, max_iter=50000, random_state=RANDOM_STATE),
    param_grid=elastic_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_linear,
    region_labels=region_train,
    results=all_results,
)

# KNN is expensive and often weak in high-dimensional sparse data, so we use only two robust target variants.
all_results = run_grid(
    model_family="KNN",
    estimator_factory=lambda **params: KNeighborsRegressor(**params),
    param_grid=knn_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=["raw", "winsorized"],
    region_labels=region_train,
    results=all_results,
)

# LinearSVR replaces slow SVR(kernel='linear').
all_results = run_grid(
    model_family="LinearSVR",
    estimator_factory=lambda **params: LinearSVR(**params, random_state=RANDOM_STATE, max_iter=50000, tol=1e-4),
    param_grid=linear_svr_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_linear,
    region_labels=region_train,
    results=all_results,
)

all_results = run_grid(
    model_family="SVR_RBF",
    estimator_factory=lambda **params: SVR(kernel="rbf", cache_size=1000, **params),
    param_grid=svr_rbf_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_svr,
    region_labels=region_train,
    results=all_results,
)

all_results = run_grid(
    model_family="SVR_Poly",
    estimator_factory=lambda **params: SVR(kernel="poly", cache_size=1000, **params),
    param_grid=svr_poly_grid,
    X_train_base=X_train_full,
    X_val_base=X_val_full,
    feature_set="FULL",
    target_variants=target_variants_svr,
    region_labels=region_train,
    results=all_results,
)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("best_val_rmse", na_position="last")
results_df.to_csv(MODELING_DIR / "model_search_results.csv", index=False)

display(results_df.head(20)[[
    "model_family", "feature_set", "target_variant", "params",
    "cv_rmse_mean", "best_val_rmse", "best_correction", "best_factor",
    "n_features", "n_train_rows", "fit_seconds"
]])


=== OLS: 4 candidates ===
[001/004] OLS | raw | {}
    CV RMSE: 103,349 | Val RMSE: 47,588 | correction=validation_multiplier
[002/004] OLS | winsorized | {}
    CV RMSE: 102,956 | Val RMSE: 47,109 | correction=validation_multiplier
[003/004] OLS | floor500 | {}
    CV RMSE: 102,522 | Val RMSE: 45,574 | correction=validation_multiplier
[004/004] OLS | drop_lt500 | {}
    CV RMSE: 89,938 | Val RMSE: 42,528 | correction=validation_multiplier

=== Ridge: 16 candidates ===
[001/016] Ridge | raw | {'alpha': 0.1}
    CV RMSE: 117,072 | Val RMSE: 53,518 | correction=validation_multiplier
[002/016] Ridge | raw | {'alpha': 0.3}
    CV RMSE: 116,965 | Val RMSE: 53,471 | correction=validation_multiplier
[003/016] Ridge | raw | {'alpha': 1.0}
    CV RMSE: 116,634 | Val RMSE: 53,314 | correction=validation_multiplier
[004/016] Ridge | raw | {'alpha': 3.0}
    CV RMSE: 115,920 | Val RMSE: 52,919 | correction=validation_multiplier
[005/016] Ridge | raw | {'alpha': 10.0}
    CV RMSE: 114,440 | Val RM

c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_clas

    CV RMSE: 101,692 | Val RMSE: 41,431 | correction=validation_multiplier
[002/012] LinearSVR | raw | {'C': 0.1, 'epsilon': 0.1}


c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_clas

    CV RMSE: 101,766 | Val RMSE: 41,373 | correction=validation_multiplier
[003/012] LinearSVR | raw | {'C': 0.1, 'epsilon': 0.2}


...

c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_clas

    CV RMSE: 101,643 | Val RMSE: 41,599 | correction=validation_multiplier
[012/012] LinearSVR | winsorized | {'C': 1.0, 'epsilon': 0.2}


c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
c:\Users\natal\anaconda3\Lib\site-packages\sklearn\svm\_clas

    CV RMSE: 102,023 | Val RMSE: 40,916 | correction=validation_multiplier

=== SVR_RBF: 16 candidates ===
[001/016] SVR_RBF | raw | {'C': 1.0, 'epsilon': 0.1, 'gamma': 'scale'}
    CV RMSE: 97,916 | Val RMSE: 37,605 | correction=validation_multiplier
[002/016] SVR_RBF | raw | {'C': 1.0, 'epsilon': 0.1, 'gamma': 0.0003}
    CV RMSE: 97,056 | Val RMSE: 37,900 | correction=validation_multiplier
[003/016] SVR_RBF | raw | {'C': 1.0, 'epsilon': 0.2, 'gamma': 'scale'}
    CV RMSE: 98,226 | Val RMSE: 37,653 | correction=validation_multiplier
[004/016] SVR_RBF | raw | {'C': 1.0, 'epsilon': 0.2, 'gamma': 0.0003}
    CV RMSE: 97,311 | Val RMSE: 37,866 | correction=validation_multiplier
[005/016] SVR_RBF | raw | {'C': 3.0, 'epsilon': 0.1, 'gamma': 'scale'}
    CV RMSE: 100,294 | Val RMSE: 39,912 | correction=validation_multiplier
[006/016] SVR_RBF | raw | {'C': 3.0, 'epsilon': 0.1, 'gamma': 0.0003}
    CV RMSE: 97,673 | Val RMSE: 38,331 | correction=validation_multiplier
[007/016] SVR_RBF | raw |

,model_family,feature_set,target_variant,params,cv_rmse_mean,best_val_rmse,best_correction,best_factor,n_features,n_train_rows,fit_seconds
95,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.293770,37603.916827,validation_multiplier,1.141892,303,1758,7.042706
87,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.566376,37604.947764,validation_multiplier,1.136937,303,1758,6.485975
97,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98227.563730,37649.682475,validation_multiplier,1.186486,303,1758,5.815400
89,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98226.199589,37652.795607,validation_multiplier,1.181532,303,1758,5.840445
90,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.947742,37865.509128,validation_multiplier,1.072523,303,1758,5.430635
98,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.618981,37874.121770,validation_multiplier,1.072523,303,1758,5.467431
88,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.175055,37900.301349,validation_multiplier,1.052703,303,1758,6.428595
96,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.441555,37900.995385,validation_multiplier,1.052703,303,1758,6.607925
92,SVR_RBF,FULL,raw,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.797677,38330.867240,validation_multiplier,0.983333,303,1758,6.590415
100,SVR_RBF,FULL,winsorized,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.411567,38331.238148,validation_multiplier,0.983333,303,1758,6.620749


## 9. LASSO-selected SVR experiment

This tests the critique-based idea that SVR may work better on fewer, more relevant features.

Procedure:

1. Train LASSO on the internal training split.
2. Keep features with non-zero LASSO coefficients.
3. Train SVR on the selected feature subset.
4. Compare validation RMSE with the full-feature SVR.

This is still ML1-compliant because LASSO and SVR are both course methods, and feature selection is allowed.

In [10]:
# ===============================================================
# 10. LASSO-selected SVR experiment
# ===============================================================

def get_lasso_selected_columns(
    X: pd.DataFrame,
    y_log: pd.Series,
    alpha: float,
    max_features: Optional[int] = None,
    min_features: int = 5,
) -> List[str]:
    """Fit LASSO and return selected columns. If too few selected, keep strongest coefficients."""
    selector = Lasso(alpha=alpha, max_iter=50000, random_state=RANDOM_STATE)
    selector.fit(X, y_log)
    coef = pd.Series(selector.coef_, index=X.columns)
    selected = coef[coef.abs() > 1e-8].abs().sort_values(ascending=False)

    if len(selected) < min_features:
        # Fallback: keep strongest coefficients even if threshold produced too few.
        selected = coef.abs().sort_values(ascending=False).head(min_features)
    if max_features is not None and len(selected) > max_features:
        selected = selected.head(max_features)
    return selected.index.tolist()


def evaluate_lasso_selected_svr(
    target_variant: str,
    lasso_alpha: float,
    max_features: Optional[int],
    svr_kernel: str,
    svr_params: Dict[str, Any],
    results: List[Dict[str, Any]],
) -> None:
    Xtr_base, ylog, yraw, reg = get_target_variant_data(X_train_full, target_variant, region_train)
    selected_cols = get_lasso_selected_columns(
        Xtr_base,
        ylog,
        alpha=lasso_alpha,
        max_features=max_features,
        min_features=10,
    )

    Xtr_sel = Xtr_base[selected_cols]
    Xval_sel = X_val_full[selected_cols]

    if svr_kernel == "rbf":
        model = SVR(kernel="rbf", cache_size=1000, **svr_params)
        family = "LASSO_Selected_SVR_RBF"
    elif svr_kernel == "poly":
        model = SVR(kernel="poly", cache_size=1000, **svr_params)
        family = "LASSO_Selected_SVR_Poly"
    else:
        raise ValueError("svr_kernel must be 'rbf' or 'poly'.")

    params_full = {
        "lasso_alpha": lasso_alpha,
        "max_features": max_features,
        "n_selected": len(selected_cols),
        **svr_params,
    }

    model_name = f"{family} | {target_variant} | {params_full}"
    print(model_name)

    start = time.time()
    cv_mean, cv_std = cv_usd_rmse(model, Xtr_sel, ylog, yraw, reg)
    report = calibration_report(model, Xtr_sel, ylog, yraw, Xval_sel, y_val_raw)
    elapsed = time.time() - start

    record = {
        "model_name": model_name,
        "model_family": family,
        "feature_set": "LASSO_SELECTED_FULL",
        "target_variant": target_variant,
        "params": json.dumps(params_full),
        "cv_rmse_mean": cv_mean,
        "cv_rmse_std": cv_std,
        "val_rmse_none": report["val_rmse_none"],
        "val_rmse_smearing": report["val_rmse_smearing"],
        "val_rmse_validation_multiplier": report["val_rmse_validation_multiplier"],
        "best_correction": report["best_correction"],
        "best_factor": report["best_factor"],
        "best_val_rmse": report["best_val_rmse"],
        "n_features": len(selected_cols),
        "n_train_rows": len(Xtr_sel),
        "fit_seconds": elapsed,
        "selected_features": ";".join(selected_cols),
    }
    results.append(record)
    print(f"    selected={len(selected_cols)} | CV RMSE={cv_mean:,.0f} | Val RMSE={record['best_val_rmse']:,.0f}")


lasso_selected_results = []

# This experiment is important, but it can become expensive.
# We focus on a compact, defensible search; expand these lists if you have time.
lasso_select_alphas = [0.001, 0.003, 0.01] if not FAST_MODE else [0.001, 0.003]
max_feature_options = [40, 60, None] if not FAST_MODE else [40, 60]
lasso_target_variants = ["raw", "winsorized", "floor500"] if not FAST_MODE else TARGET_VARIANTS_FAST

# Focused SVR configs based on likely useful ranges.
rbf_configs = [
    {"C": 1.0, "gamma": "scale", "epsilon": 0.1},
    {"C": 3.0, "gamma": "scale", "epsilon": 0.1},
    {"C": 1.0, "gamma": 0.0003, "epsilon": 0.1},
]
if FAST_MODE:
    rbf_configs = rbf_configs[:2]

poly_configs = [
    {"C": 1.0, "degree": 2, "gamma": "scale", "epsilon": 0.1, "coef0": 1.0},
    {"C": 3.0, "degree": 2, "gamma": "scale", "epsilon": 0.1, "coef0": 1.0},
]
if FAST_MODE:
    poly_configs = poly_configs[:1]

for target_variant in lasso_target_variants:
    for alpha in lasso_select_alphas:
        for max_features in max_feature_options:
            for params in rbf_configs:
                evaluate_lasso_selected_svr(target_variant, alpha, max_features, "rbf", params, lasso_selected_results)
            for params in poly_configs:
                evaluate_lasso_selected_svr(target_variant, alpha, max_features, "poly", params, lasso_selected_results)
            pd.DataFrame(lasso_selected_results).to_csv(MODELING_DIR / "lasso_selected_svr_results_partial.csv", index=False)

lasso_selected_df = pd.DataFrame(lasso_selected_results)
lasso_selected_df.to_csv(MODELING_DIR / "lasso_selected_svr_results.csv", index=False)

def load_or_empty(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

main_results_df = load_or_empty(MODELING_DIR / "model_search_results.csv")
combined_results_df = pd.concat([main_results_df, lasso_selected_df], ignore_index=True, sort=False)
combined_results_df = combined_results_df.sort_values("best_val_rmse", na_position="last")
combined_results_df.to_csv(MODELING_DIR / "all_model_results.csv", index=False)

display(combined_results_df.head(30)[[
    "model_family", "feature_set", "target_variant", "params",
    "cv_rmse_mean", "best_val_rmse", "best_correction", "best_factor", "n_features", "n_train_rows"
]])

LASSO_Selected_SVR_RBF | raw | {'lasso_alpha': 0.001, 'max_features': 40, 'n_selected': 40, 'C': 1.0, 'gamma': 'scale', 'epsilon': 0.1}
    selected=40 | CV RMSE=97,831 | Val RMSE=41,313
LASSO_Selected_SVR_RBF | raw | {'lasso_alpha': 0.001, 'max_features': 40, 'n_selected': 40, 'C': 3.0, 'gamma': 'scale', 'epsilon': 0.1}
    selected=40 | CV RMSE=99,260 | Val RMSE=43,874
LASSO_Selected_SVR_Poly | raw | {'lasso_alpha': 0.001, 'max_features': 40, 'n_selected': 40, 'C': 1.0, 'degree': 2, 'gamma': 'scale', 'epsilon': 0.1, 'coef0': 1.0}
    selected=40 | CV RMSE=99,849 | Val RMSE=42,213
LASSO_Selected_SVR_RBF | raw | {'lasso_alpha': 0.001, 'max_features': 60, 'n_selected': 60, 'C': 1.0, 'gamma': 'scale', 'epsilon': 0.1}
    selected=60 | CV RMSE=97,760 | Val RMSE=40,822
LASSO_Selected_SVR_RBF | raw | {'lasso_alpha': 0.001, 'max_features': 60, 'n_selected': 60, 'C': 3.0, 'gamma': 'scale', 'epsilon': 0.1}
    selected=60 | CV RMSE=99,527 | Val RMSE=44,460
LASSO_Selected_SVR_Poly | raw | {'las

,model_family,feature_set,target_variant,params,cv_rmse_mean,best_val_rmse,best_correction,best_factor,n_features,n_train_rows
0,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.293770,37603.916827,validation_multiplier,1.141892,303,1758
1,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.566376,37604.947764,validation_multiplier,1.136937,303,1758
2,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98227.563730,37649.682475,validation_multiplier,1.186486,303,1758
3,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98226.199589,37652.795607,validation_multiplier,1.181532,303,1758
4,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.947742,37865.509128,validation_multiplier,1.072523,303,1758
5,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.618981,37874.121770,validation_multiplier,1.072523,303,1758
6,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.175055,37900.301349,validation_multiplier,1.052703,303,1758
7,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.441555,37900.995385,validation_multiplier,1.052703,303,1758
8,SVR_RBF,FULL,raw,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.797677,38330.867240,validation_multiplier,0.983333,303,1758
9,SVR_RBF,FULL,winsorized,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.411567,38331.238148,validation_multiplier,0.983333,303,1758


## 10. Select the best model using validation RMSE

We select by the lowest validation RMSE on the original USD scale. The internal test set is still untouched at this point.

In [11]:
# ===============================================================
# 11. Select best model from validation results
# ===============================================================

results_path = MODELING_DIR / "all_model_results.csv"
if not results_path.exists():
    results_path = MODELING_DIR / "model_search_results.csv"

results_df = pd.read_csv(results_path)
results_df = results_df.dropna(subset=["best_val_rmse"]).sort_values("best_val_rmse")

# Baselines remain in the comparison table, but the final selected model should be
# an actual ML1 algorithm unless every model failed.
model_candidates_df = results_df[results_df["model_family"] != "baseline"].copy()
if model_candidates_df.empty:
    raise RuntimeError("No non-baseline model candidates are available. Check earlier search errors.")

best_row = model_candidates_df.iloc[0].to_dict()
print("Best validation model:")
for key in ["model_family", "feature_set", "target_variant", "params", "cv_rmse_mean", "best_val_rmse", "best_correction", "best_factor", "n_features", "n_train_rows"]:
    print(f"{key}: {best_row.get(key)}")

# Save top table.
top_models = results_df.head(30).copy()
top_models.to_csv(MODELING_DIR / "top_validation_models.csv", index=False)
display(top_models[[
    "model_family", "feature_set", "target_variant", "params",
    "cv_rmse_mean", "best_val_rmse", "best_correction", "best_factor", "n_features", "n_train_rows"
]])

with open(MODELING_DIR / "best_validation_model.json", "w", encoding="utf-8") as f:
    json.dump(best_row, f, indent=2)

Best validation model:
model_family: SVR_RBF
feature_set: FULL
target_variant: winsorized
params: {"C": 1.0, "epsilon": 0.1, "gamma": "scale"}
cv_rmse_mean: 97915.29377004252
best_val_rmse: 37603.91682650413
best_correction: validation_multiplier
best_factor: 1.141891891891892
n_features: 303
n_train_rows: 1758


,model_family,feature_set,target_variant,params,cv_rmse_mean,best_val_rmse,best_correction,best_factor,n_features,n_train_rows
0,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.293770,37603.916827,validation_multiplier,1.141892,303,1758
1,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": ""scale""}",97915.566376,37604.947764,validation_multiplier,1.136937,303,1758
2,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98227.563730,37649.682475,validation_multiplier,1.186486,303,1758
3,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": ""scale""}",98226.199589,37652.795607,validation_multiplier,1.181532,303,1758
4,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.947742,37865.509128,validation_multiplier,1.072523,303,1758
5,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.2, ""gamma"": 0.0003}",97310.618981,37874.121770,validation_multiplier,1.072523,303,1758
6,SVR_RBF,FULL,raw,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.175055,37900.301349,validation_multiplier,1.052703,303,1758
7,SVR_RBF,FULL,winsorized,"{""C"": 1.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97056.441555,37900.995385,validation_multiplier,1.052703,303,1758
8,SVR_RBF,FULL,raw,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.797677,38330.867240,validation_multiplier,0.983333,303,1758
9,SVR_RBF,FULL,winsorized,"{""C"": 3.0, ""epsilon"": 0.1, ""gamma"": 0.0003}",97672.411567,38331.238148,validation_multiplier,0.983333,303,1758


## 11. Reconstruct and evaluate the selected model on the internal test set

This is the first time we use the internal test set for the selected model. This gives our best estimate of private leaderboard RMSE.

In [12]:
# ===============================================================
# 12. Reconstruct selected model and evaluate on internal test
# ===============================================================

def parse_params(params_json: str) -> Dict[str, Any]:
    if isinstance(params_json, dict):
        return params_json
    if pd.isna(params_json) or params_json == "{}":
        return {}
    return json.loads(params_json)


def build_estimator_from_row(row: Dict[str, Any]):
    family = row["model_family"]
    params = parse_params(row.get("params", "{}"))

    # Remove feature-selection-only params if present.
    estimator_params = dict(params)
    for key in ["lasso_alpha", "max_features", "n_selected"]:
        estimator_params.pop(key, None)

    if family == "OLS":
        return LinearRegression(**estimator_params)
    if family == "Ridge":
        return Ridge(**estimator_params, random_state=RANDOM_STATE)
    if family == "LASSO":
        return Lasso(**estimator_params, max_iter=50000, random_state=RANDOM_STATE)
    if family == "ElasticNet":
        return ElasticNet(**estimator_params, max_iter=50000, random_state=RANDOM_STATE)
    if family == "KNN":
        return KNeighborsRegressor(**estimator_params)
    if family == "LinearSVR":
        return LinearSVR(**estimator_params, random_state=RANDOM_STATE, max_iter=50000, tol=1e-4)
    if family == "SVR_RBF":
        return SVR(kernel="rbf", cache_size=1000, **estimator_params)
    if family == "SVR_Poly":
        return SVR(kernel="poly", cache_size=1000, **estimator_params)
    if family == "LASSO_Selected_SVR_RBF":
        return SVR(kernel="rbf", cache_size=1000, **estimator_params)
    if family == "LASSO_Selected_SVR_Poly":
        return SVR(kernel="poly", cache_size=1000, **estimator_params)

    raise ValueError(f"Cannot reconstruct model family: {family}")


def get_feature_matrices_for_row(row: Dict[str, Any]):
    """Return X_train, X_val, X_internal_test, X_kaggle, region labels, and selected cols if needed."""
    feature_set = row["feature_set"]
    family = row["model_family"]

    if feature_set == "OLS_SAFE":
        return X_train_ols, X_val_ols, X_internal_test_ols, None, region_train, None

    if feature_set == "FULL":
        return X_train_full, X_val_full, X_internal_test_full, X_kaggle_full, region_train, None

    if feature_set == "LASSO_SELECTED_FULL":
        if "selected_features" in row and isinstance(row["selected_features"], str) and len(row["selected_features"]) > 0:
            selected_cols = row["selected_features"].split(";")
        else:
            params = parse_params(row["params"])
            alpha = float(params["lasso_alpha"])
            max_features = params.get("max_features", None)
            if max_features is not None:
                max_features = int(max_features)
            Xtr_tmp, ylog_tmp, _, _ = get_target_variant_data(X_train_full, row["target_variant"], region_train)
            selected_cols = get_lasso_selected_columns(Xtr_tmp, ylog_tmp, alpha=alpha, max_features=max_features, min_features=10)

        return (
            X_train_full[selected_cols],
            X_val_full[selected_cols],
            X_internal_test_full[selected_cols],
            X_kaggle_full[selected_cols],
            region_train,
            selected_cols,
        )

    raise ValueError(f"Unknown feature set: {feature_set}")

selected_model = build_estimator_from_row(best_row)
Xtr_base, Xval_base, Xtest_base, Xkaggle_base, reg_labels, selected_cols = get_feature_matrices_for_row(best_row)
Xtr, ylog, yraw, reg = get_target_variant_data(Xtr_base, best_row["target_variant"], reg_labels)

selected_model.fit(Xtr, ylog)

pred_val_log = np.asarray(selected_model.predict(Xval_base), dtype=float)
pred_test_log = np.asarray(selected_model.predict(Xtest_base), dtype=float)

# Use the correction selected on validation.
correction = best_row["best_correction"]
factor = float(best_row["best_factor"])

# For smearing, recompute the smearing factor on the selected training data.
if correction == "train_smearing":
    pred_train_log = np.asarray(selected_model.predict(Xtr), dtype=float)
    factor = float(np.mean(np.exp(np.asarray(ylog) - pred_train_log)))

val_rmse_check = usd_rmse_from_log(y_val_raw, pred_val_log, factor=factor)
internal_test_rmse = usd_rmse_from_log(y_internal_test_raw, pred_test_log, factor=factor)

internal_test_summary = {
    "model_family": best_row["model_family"],
    "feature_set": best_row["feature_set"],
    "target_variant": best_row["target_variant"],
    "params": best_row["params"],
    "best_correction": correction,
    "factor_used": factor,
    "validation_rmse_recomputed": val_rmse_check,
    "internal_test_rmse": internal_test_rmse,
    "n_features": int(Xtr.shape[1]),
    "n_train_rows": int(Xtr.shape[0]),
}

print("Selected model internal-test evaluation:")
for key, value in internal_test_summary.items():
    print(f"{key}: {value}")

with open(MODELING_DIR / "internal_test_summary.json", "w", encoding="utf-8") as f:
    json.dump(internal_test_summary, f, indent=2)

pd.DataFrame([internal_test_summary]).to_csv(MODELING_DIR / "internal_test_summary.csv", index=False)

Selected model internal-test evaluation:
model_family: SVR_RBF
feature_set: FULL
target_variant: winsorized
params: {"C": 1.0, "epsilon": 0.1, "gamma": "scale"}
best_correction: validation_multiplier
factor_used: 1.141891891891892
validation_rmse_recomputed: 37603.91682650413
internal_test_rmse: 52217.00158081891
n_features: 303
n_train_rows: 1758


## 12. Error analysis for the selected model

This helps explain where the model performs well or poorly.

In [13]:
# ===============================================================
# 13. Error analysis
# ===============================================================

pred_test_usd = safe_exp_to_usd(pred_test_log, factor=factor)
errors = pred_test_usd - y_internal_test_raw.to_numpy()
abs_errors = np.abs(errors)

error_df = pd.DataFrame({
    "actual_salary": y_internal_test_raw,
    "predicted_salary": pred_test_usd,
    "error": errors,
    "absolute_error": abs_errors,
    "region": region_internal_test,
})

# Salary buckets.
error_df["salary_bucket"] = pd.cut(
    error_df["actual_salary"],
    bins=[0, 500, 10_000, 30_000, 60_000, 100_000, 200_000, np.inf],
    labels=["<500", "500-10k", "10k-30k", "30k-60k", "60k-100k", "100k-200k", "200k+"],
    include_lowest=True,
)

bucket_summary = error_df.groupby("salary_bucket", observed=False).agg(
    n=("actual_salary", "size"),
    mean_actual=("actual_salary", "mean"),
    mean_pred=("predicted_salary", "mean"),
    rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
    mae=("absolute_error", "mean"),
).reset_index()

region_summary = error_df.groupby("region").agg(
    n=("actual_salary", "size"),
    rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
    mae=("absolute_error", "mean"),
    mean_actual=("actual_salary", "mean"),
    mean_pred=("predicted_salary", "mean"),
).sort_values("n", ascending=False).reset_index()

print("Overall internal-test RMSE:", f"${internal_test_rmse:,.0f}")
print("Overall internal-test MAE:", f"${mean_absolute_error(y_internal_test_raw, pred_test_usd):,.0f}")

display(bucket_summary)
display(region_summary.head(20))

error_df.to_csv(MODELING_DIR / "selected_model_internal_test_predictions.csv", index=False)
bucket_summary.to_csv(MODELING_DIR / "error_by_salary_bucket.csv", index=False)
region_summary.to_csv(MODELING_DIR / "error_by_region.csv", index=False)

Overall internal-test RMSE: $52,217
Overall internal-test MAE: $24,036


,salary_bucket,n,mean_actual,mean_pred,rmse,mae
0,<500,12,249.250000,28520.797580,29463.653532,28271.547580
1,500-10k,58,2718.844828,36253.886884,38221.509789,33535.042057
2,10k-30k,73,21180.684932,29401.760855,15371.447847,11519.100867
3,30k-60k,137,45077.379562,48778.543170,20069.074514,14697.527245
4,60k-100k,74,74173.378378,60145.893782,27871.117686,22591.617116
5,100k-200k,21,132373.000000,73034.078590,67195.562481,61627.019453
6,200k+,2,557293.000000,78845.531578,595001.859465,478447.468422


,region,n,rmse,mae,mean_actual,mean_pred
0,R17,166,30702.441718,22767.153771,32378.680723,36569.866775
1,R11,65,37171.917516,25891.999649,59985.738462,59604.857194
2,R13,32,19754.475361,15230.631998,43421.812500,48197.736259
3,R18,18,33193.828767,23174.107087,55454.166667,46077.310203
4,R03,17,202583.359280,62553.520273,98686.470588,48443.868776
5,R05,12,23216.670260,17715.529403,54507.166667,64739.987952
6,R06,11,30005.597603,26600.752237,43516.545455,49292.927510
7,R16,9,31948.635836,22843.277984,51722.666667,46063.123695
8,R09,9,22247.107880,18640.333964,52779.333333,59539.849176
9,R08,9,26201.635256,23248.498811,45397.555556,58306.937504


## 13. Generate Kaggle submission

This produces a submission using the selected model trained on the internal training split. This is the safest version directly compatible with `outputs_v6`.

After model selection, you may later create a separate final-refit pipeline that re-runs preprocessing on all labeled data. For now, this file is sufficient to submit and check leaderboard behavior.

In [14]:
# ===============================================================
# 14. Generate Kaggle submission from selected model
# ===============================================================

if Xkaggle_base is None:
    print("Selected model uses OLS_SAFE; loading Kaggle OLS_SAFE matrix.")
    Xkaggle_base = pd.read_csv(OUTPUT_DIR / "X_kaggle_ols_safe_scaled.csv")

# selected_model is already fitted on selected training data above.
pred_kaggle_log = np.asarray(selected_model.predict(Xkaggle_base), dtype=float)
pred_kaggle_usd = safe_exp_to_usd(pred_kaggle_log, factor=factor)

submission = pd.DataFrame({
    "id": kaggle_ids.astype(int),
    "annual.pay.usd": pred_kaggle_usd.astype(float),
})

# Basic sanity checks.
assert submission.shape[0] == len(kaggle_ids), "Submission row count mismatch."
assert submission["id"].is_unique, "Submission IDs are not unique."
assert submission["annual.pay.usd"].isna().sum() == 0, "Submission has missing predictions."
assert (submission["annual.pay.usd"] > 0).all(), "Submission has non-positive predictions."

submission_path = MODELING_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission saved to:", submission_path.resolve())
display(submission.head())
display(submission["annual.pay.usd"].describe().to_frame())

Submission saved to: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\modeling_outputs_v6\submission.csv


,id,annual.pay.usd
0,0,17812.025003
1,1,38336.888569
2,2,59658.806765
3,3,66937.582005
4,4,27105.963156


,annual.pay.usd
count,628.000000
mean,45798.250624
std,24184.193377
min,5462.028875
25%,29296.836831
50%,41853.067548
75%,56820.693563
max,241677.392462


## 14. Optional: final-refit guidance

The safest final competition workflow is:

1. Use this notebook to select the best model and hyperparameters.
2. Report validation RMSE and internal-test RMSE in the presentation.
3. For the final Kaggle submission, optionally create a separate final-refit notebook that:
   - locks the chosen model,
   - re-runs preprocessing on a larger labeled set,
   - trains the chosen model once,
   - generates `submission.csv`.

Do not use internal-test performance to keep tuning. Once internal test is checked, the model choice should be considered locked.

In [15]:
# ===============================================================
# 15. Save final modeling summary
# ===============================================================

final_summary = {
    "selected_model": internal_test_summary,
    "feature_engineering_summary": {
        "feature_shapes": feature_summary.get("feature_shapes"),
        "target_policy": feature_summary.get("target_policy"),
        "low_salary_policy": feature_summary.get("low_salary_policy", None),
    },
    "files_created": {
        "all_model_results": str((MODELING_DIR / "all_model_results.csv").resolve()) if (MODELING_DIR / "all_model_results.csv").exists() else None,
        "top_validation_models": str((MODELING_DIR / "top_validation_models.csv").resolve()),
        "internal_test_summary": str((MODELING_DIR / "internal_test_summary.csv").resolve()),
        "submission": str((MODELING_DIR / "submission.csv").resolve()),
    },
    "important_note": "Model selected by validation RMSE. Internal test used once for final performance estimate.",
}

with open(MODELING_DIR / "final_modeling_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("Final modeling summary saved.")
print(json.dumps(final_summary, indent=2)[:2000])

Final modeling summary saved.
{
  "selected_model": {
    "model_family": "SVR_RBF",
    "feature_set": "FULL",
    "target_variant": "winsorized",
    "params": "{\"C\": 1.0, \"epsilon\": 0.1, \"gamma\": \"scale\"}",
    "best_correction": "validation_multiplier",
    "factor_used": 1.141891891891892,
    "validation_rmse_recomputed": 37603.91682650413,
    "internal_test_rmse": 52217.00158081891,
    "n_features": 303,
    "n_train_rows": 1758
  },
  "feature_engineering_summary": {
    "feature_shapes": {
      "full_scaled": [
        1758,
        303
      ],
      "ols_safe_scaled": [
        1758,
        108
      ]
    },
    "target_policy": {
      "validation_and_internal_test_targets": "raw_not_cleaned_not_winsorized",
      "available_training_targets": [
        "y_train_log_raw.csv",
        "y_train_log_winsorized.csv",
        "y_train_log_floor500.csv",
        "train_keep_salary_ge_500_mask.csv"
      ],
      "low_salary_floor_usd": 500.0,
      "low_salary_counts